# Sincronização permanente: Google Drive ↔ GitHub

Este notebook mantém uma cópia do **[vision-transfer-learning](https://github.com/Gengo250/vision-transfer-learning)** no **seu Google Drive**, não em uma pasta de trabalho temporária.

1. Execute a célula abaixo e autorize o acesso ao Google Drive.
2. O repositório ficará permanentemente em **Meu Drive/vision-transfer-learning** (caminho no Colab: `/content/drive/MyDrive/vision-transfer-learning`).
3. Na primeira execução, a célula faz `git clone`; nas próximas, `git pull --ff-only` na branch `main`.
4. Execute a célula final para verificar o commit e os arquivos disponíveis.

> `/content/drive` é somente o ponto de montagem do Google Drive. Os arquivos guardados em **MyDrive** permanecem na sua conta mesmo após o encerramento da sessão Colab.

> O `git pull` sincroniza **GitHub → Google Drive**. Mudanças locais no clone não são enviadas automaticamente ao GitHub. O notebook interrompe a atualização se detectar mudanças locais, evitando sobrescrevê-las. Para publicar mudanças de código feitas no clone, será necessário executar `git add/commit/push` com autenticação apropriada. A opção **Arquivo → Salvar uma cópia no GitHub** salva o notebook aberto no GitHub, mas não faz push de todos os arquivos modificados no clone.


In [ ]:
from pathlib import Path
import os
import subprocess
from google.colab import drive

# Monta o Google Drive: o conteudo em MyDrive e persistente.
drive.mount("/content/drive")

REPO_URL = "https://github.com/Gengo250/vision-transfer-learning.git"
BRANCH = "main"
REPO_DIR = Path("/content/drive/MyDrive/vision-transfer-learning")

def git(*args, cwd=None):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} existe, mas nao e um repositorio Git.")

    current_branch = git("branch", "--show-current", cwd=REPO_DIR)
    if current_branch != BRANCH:
        raise RuntimeError(
            f"Branch atual: {current_branch}. Esperada: {BRANCH}. "
            "Troque de branch manualmente antes de atualizar."
        )

    changes = git("status", "--porcelain", cwd=REPO_DIR)
    if changes:
        raise RuntimeError(
            "Ha alteracoes locais no clone do Colab. "
            "Salve-as antes de executar git pull para evitar perda de trabalho.\n"
            + changes
        )

    print("Atualizando o repositorio...")
    print(git("pull", "--ff-only", "origin", BRANCH, cwd=REPO_DIR))
else:
    print("Clonando o repositorio...")
    print(git("clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)))

os.chdir(REPO_DIR)
print(f"\nDiretorio: {Path.cwd()}")
print("Commit:", git("log", "-1", "--format=%h %s", cwd=REPO_DIR))


## Conferir clone persistente

Verifique o caminho permanente, a branch e o estado do repositório.


In [ ]:
print("Branch:", git("branch", "--show-current", cwd=REPO_DIR))
print("Status:")
print(git("status", "--short", cwd=REPO_DIR) or "(sem alteracoes locais)")
print("\nArquivos na raiz:")
for path in sorted(REPO_DIR.iterdir()):
    if path.name != ".git":
        print("📁" if path.is_dir() else "📄", path.name)


## Publicar alterações do Drive no GitHub

Use esta etapa **depois** de editar e salvar os arquivos que estão dentro de `Meu Drive/vision-transfer-learning`.

**Configure uma única vez:** no Colab, abra o painel **Segredos** (ícone de chave), crie o segredo `GITHUB_TOKEN` com um token de acesso pessoal *fine-grained* do GitHub, limitado ao repositório `Gengo250/vision-transfer-learning` e com permissão **Contents: Read and write**. Autorize o acesso desse notebook ao segredo. **Nunca escreva o token em uma célula ou arquivo do projeto.**

Na célula a seguir, revise os arquivos alterados, informe uma mensagem de commit e confirme digitando `PUBLICAR`. Ela executa `git add`, `git commit` e `git push origin main`. **Não acontece automaticamente ao salvar.** O GitHub não recebe arquivos ignorados pelo `.gitignore`, como `data/` e `runs/results/`.

> Atenção: salve o notebook atual antes de publicar. Para editar o próprio `colab.ipynb`, abra a versão que está dentro de `Meu Drive/vision-transfer-learning`, não uma visualização avulsa carregada do GitHub. Mudanças concorrentes feitas em outra máquina podem exigir atualizar ou reconciliar a branch antes do push.


In [ ]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

REPO_DIR = Path("/content/drive/MyDrive/vision-transfer-learning")
assert (REPO_DIR / ".git").is_dir(), "Execute a sincronizacao antes."
assert git("branch", "--show-current", cwd=REPO_DIR) == "main", "Esperada branch main."

status = git("status", "--short", cwd=REPO_DIR)
print("Arquivos alterados:")
print(status or "(nenhuma alteracao pendente)")

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Configure o segredo GITHUB_TOKEN no painel Segredos do Colab.")

if status:
    mensagem = input("Mensagem do commit: ").strip()
    if not mensagem:
        raise ValueError("A mensagem de commit nao pode estar vazia.")

    email = subprocess.run(
        ["git", "config", "--local", "--get", "user.email"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    if not email:
        email = input("Email de autoria dos commits do GitHub: ").strip()
        if not email:
            raise ValueError("Informe um email valido para o autor do commit.")
        git("config", "--local", "user.name", "Gengo250", cwd=REPO_DIR)
        git("config", "--local", "user.email", email, cwd=REPO_DIR)

    print("\\nATENCAO: todos os arquivos nao ignorados acima serao incluidos.")
    if input('Digite PUBLICAR para confirmar: ').strip() != "PUBLICAR":
        raise RuntimeError("Publicacao cancelada; nenhum commit foi criado.")

    git("add", "--all", cwd=REPO_DIR)
    git("commit", "-m", mensagem, cwd=REPO_DIR)
else:
    if input('Sem novos arquivos. Digite PUBLICAR para enviar commits locais pendentes: ').strip() != "PUBLICAR":
        raise RuntimeError("Publicacao cancelada.")

# Segredo somente no ambiente do processo; nao grava token no Git ou no notebook.
with tempfile.NamedTemporaryFile(mode="w", suffix=".sh", prefix="git_askpass_",
                                 dir="/tmp", delete=False) as script:
    script.write('#!/bin/sh\\ncase "$1" in\\n  *Username*) printf "x-access-token\\\\n";;\\n  *Password*) printf "%s\\\\n" "$GH_TOKEN";;\\nesac\\n')
    askpass_path = script.name

try:
    os.chmod(askpass_path, 0o700)
    env = os.environ.copy()
    env["GH_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass_path
    env["GIT_TERMINAL_PROMPT"] = "0"
    result = subprocess.run(
        ["git", "-c", "credential.helper=", "push", "origin", "main"],
        cwd=REPO_DIR, env=env, capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError("git push falhou: " + (result.stderr or result.stdout))
    print("GitHub atualizado com sucesso!")
    print(git("log", "-1", "--format=%h %s", cwd=REPO_DIR))
finally:
    os.unlink(askpass_path)
    del token
